# H3 — Synthetic Control: Indie vs AAA Heterogeneity

**Hypothesis:** The marginal review lift from Winter 2024 sale participation is larger for indie games than AAA games.

**Method:** Synthetic control with placebo-based inference. 5 indie + 5 AAA treated games, donor pool = same-genre non-treated games.

In [ ]:
import os, json, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv
import psycopg

sys.path.insert(0, str(Path("..").resolve()))
from src.analysis.synthetic_control import fit_synthetic_control, placebo_test, compute_pvalue

load_dotenv()
RESULTS = Path("../results")
plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False})
conn = psycopg.connect(os.environ["DATABASE_URL"])
print("Connected.")


## 1. Build daily review panel

In [ ]:
panel = pd.read_sql("""
    SELECT r.appid, r.review_date::TEXT AS review_date, r.review_count
    FROM mart.fct_reviews_daily r
    JOIN stg.games g USING (appid)
    ORDER BY r.review_date, r.appid
""", conn)

panel["log_reviews"] = np.log1p(panel["review_count"])
print(f"Panel rows: {len(panel):,}")
print(f"Date range: {panel['review_date'].min()} – {panel['review_date'].max()}")
print(f"Unique games: {panel['appid'].nunique():,}")


## 2. Select treated and donor units

In [ ]:
tc = pd.read_parquet(RESULTS / "eda_treatment_control.parquet")

# Fetch game names for display (separate from tc merge)
games = pd.read_sql("""
    SELECT appid, name FROM stg.games
""", conn)

# tc already has is_indie, is_aaa, sale_reviews from EDA step
tc = tc.merge(games[["appid","name"]], on="appid", how="left", suffixes=("","_db"))
if "name_db" in tc.columns:
    tc["name"] = tc["name_db"].fillna(tc["name"])
    tc.drop(columns=["name_db"], inplace=True)

# Select treated games with most reviews during the sale (best signal)
treated = tc[tc["treated"]].copy()
treated_indie = (treated[treated["is_indie"] == True]
                 .sort_values("sale_reviews", ascending=False)
                 .head(5)["appid"].tolist())
treated_aaa   = (treated[treated["is_aaa"] == True]
                 .sort_values("sale_reviews", ascending=False)
                 .head(5)["appid"].tolist())

print(f"Top 5 indie treated: {treated_indie}")
print(f"Top 5 AAA treated:   {treated_aaa}")

# Donor pool: non-treated games with data in our panel
non_treated_appids = tc[~tc["treated"]]["appid"].tolist()
donor_pool = [a for a in non_treated_appids if a in panel["appid"].unique()]
print(f"Donor pool size: {len(donor_pool):,}")


## 3. Fit synthetic controls

In [ ]:
SC_KWARGS = dict(
    date_col="review_date",
    outcome_col="log_reviews",
    pre_end="2024-12-18",
    post_start="2024-12-19",
    post_end="2025-01-02",
)

name_map = dict(zip(tc["appid"], tc["name"]))

indie_results, aaa_results = [], []
for appid in treated_indie:
    name = name_map.get(appid, str(appid))
    r = fit_synthetic_control(panel, appid, donor_pool, **SC_KWARGS)
    if "error" in r:
        print(f"  {name}: {r['error']}")
    else:
        print(f"  {name}: pre_RMSPE={r['pre_rmspe']:.4f}  mean_gap={r['mean_gap_post']:+.4f}")
        indie_results.append({"name": name, **r})

print()
for appid in treated_aaa:
    name = name_map.get(appid, str(appid))
    r = fit_synthetic_control(panel, appid, donor_pool, **SC_KWARGS)
    if "error" in r:
        print(f"  {name}: {r['error']}")
    else:
        print(f"  {name}: pre_RMSPE={r['pre_rmspe']:.4f}  mean_gap={r['mean_gap_post']:+.4f}")
        aaa_results.append({"name": name, **r})


## 4. Gap plots

In [ ]:
def plot_gaps(results, title, color, ax=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(10, 4))
    for r in results:
        gs = r.get("gap_series")
        if gs is not None and len(gs):
            ax.plot(pd.to_datetime(gs.index), gs.values,
                    alpha=0.6, lw=1.5, label=r["name"][:25])
    ax.axhline(0, color="black", lw=1.5, ls="--")
    ax.axvline(pd.Timestamp("2024-12-19"), color="black", ls=":", lw=1)
    ax.set_ylabel("Actual − Synthetic (log reviews)")
    ax.set_title(title)
    ax.legend(fontsize=8)
    return ax

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
plot_gaps(indie_results, "Indie treated games", "steelblue", axes[0])
plot_gaps(aaa_results,   "AAA treated games",   "coral",     axes[1])
plt.suptitle("Synthetic control gap plots (positive = sale lift above synthetic)", y=1.02)
plt.tight_layout()
plt.savefig(RESULTS / "fig_sc_gap_plots.png", dpi=150, bbox_inches="tight")
plt.show()


## 5. Placebo test

In [ ]:
all_treated = treated_indie + treated_aaa
print("Running placebo test (n=50)...")
placebos = placebo_test(panel, all_treated, donor_pool, n_placebos=50, seed=42, **SC_KWARGS)
print(f"Placebo tests completed: {placebos['n_placebos']}")

placebo_gaps = [r["mean_gap"] for r in placebos["placebo_results"]]

# Compare treated mean gaps to placebo distribution
indie_mean_gap = np.mean([r["mean_gap_post"] for r in indie_results]) if indie_results else np.nan
aaa_mean_gap   = np.mean([r["mean_gap_post"] for r in aaa_results])   if aaa_results   else np.nan

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(placebo_gaps, bins=25, color="steelblue", edgecolor="white", alpha=0.7, label="Placebo gaps")
if not np.isnan(indie_mean_gap):
    ax.axvline(indie_mean_gap, color="green", lw=2, label=f"Indie mean gap = {indie_mean_gap:+.3f}")
if not np.isnan(aaa_mean_gap):
    ax.axvline(aaa_mean_gap, color="orange", lw=2, label=f"AAA mean gap = {aaa_mean_gap:+.3f}")
ax.axvline(0, color="black", lw=1, ls="--")
ax.set_xlabel("Mean post-sale gap (actual − synthetic)")
ax.set_title("Placebo distribution vs treated gaps")
ax.legend()
plt.tight_layout()
plt.savefig(RESULTS / "fig_sc_placebo.png", dpi=150)
plt.show()

# p-values
if not np.isnan(indie_mean_gap):
    p_indie = np.mean(np.array(placebo_gaps) >= indie_mean_gap)
    print(f"Indie mean gap: {indie_mean_gap:+.4f}  one-sided p = {p_indie:.3f}")
if not np.isnan(aaa_mean_gap):
    p_aaa = np.mean(np.array(placebo_gaps) >= aaa_mean_gap)
    print(f"AAA mean gap:   {aaa_mean_gap:+.4f}  one-sided p = {p_aaa:.3f}")


## 6. Heterogeneity table + save results

In [ ]:
het_table = []
for tier, results in [("Indie", indie_results), ("AAA", aaa_results)]:
    for r in results:
        het_table.append({
            "tier": tier,
            "name": r["name"],
            "pre_rmspe": r["pre_rmspe"],
            "mean_gap_post": r["mean_gap_post"],
            "n_donors": r["n_donors"],
        })

het_df = pd.DataFrame(het_table)
if not het_df.empty:
    print(het_df.groupby("tier")[["pre_rmspe","mean_gap_post"]].agg(["mean","std"]).round(4))

h3_results = {
    "indie_mean_gap": float(indie_mean_gap) if not np.isnan(indie_mean_gap) else None,
    "aaa_mean_gap":   float(aaa_mean_gap)   if not np.isnan(aaa_mean_gap)   else None,
    "n_indie_treated": len(indie_results),
    "n_aaa_treated":   len(aaa_results),
    "n_placebos": placebos["n_placebos"],
    "heterogeneity": het_df.to_dict(orient="records") if not het_df.empty else [],
}

import json
with open(RESULTS / "h3_sc_results.json", "w") as f:
    json.dump(h3_results, f, indent=2)
print(json.dumps({k: v for k, v in h3_results.items() if k != "heterogeneity"}, indent=2))
conn.close()
